# SpectralWaste

### Imports

In [ ]:
import os
import sys
import yaml
import torch
import numpy as np
from pathlib import Path
from collections import OrderedDict
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from copy import deepcopy
import torch.nn.functional as F
from torch.utils.data import DataLoader
from tqdm import tqdm
from contextlib import redirect_stdout
import gc

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from pyfunctions.build_model import build_model_finetune
from pyfunctions.dataload import create_datasets
from pyfunctions.metrics import calculate_metrics

print("All imports loaded successfully")

### Functions

In [ ]:
def get_palette_for_pil(num_classes=7):
    base = {
        0: [0, 0, 0],
        1: [218, 247, 6],
        2: [51, 221, 255],
        3: [52, 50, 221],
        4: [202, 152, 195],
        5: [0, 128, 0],
        6: [255, 165, 0],
    }
    palette = []
    for i in range(num_classes):
        palette.extend(base.get(i, [128, 128, 128]))
    if len(palette) < 768:
        palette.extend([0] * (768 - len(palette)))
    return palette[:768]


def denorm_image(t, modality='rgb', hsi_means=None, hsi_stds=None):
    if modality == 'rgb' and t.ndim == 3 and t.shape[0] == 3:
        mean = torch.tensor([0.485, 0.456, 0.406], device=t.device, dtype=t.dtype).view(3, 1, 1)
        std = torch.tensor([0.229, 0.224, 0.225], device=t.device, dtype=t.dtype).view(3, 1, 1)
        return torch.clamp(t * std + mean, 0, 1)
    elif modality == 'hsi' and hsi_means is not None and hsi_stds is not None:
        if len(hsi_means) == len(hsi_stds) == t.shape[0]:
            mean = torch.tensor(hsi_means, device=t.device, dtype=t.dtype).view(-1, 1, 1)
            std = torch.tensor(hsi_stds, device=t.device, dtype=t.dtype).view(-1, 1, 1)
            return t * std + mean
    return torch.clamp(t, 0, 1) if t.dtype.is_floating_point else t


def cleanup_cuda(device=None, vars_to_del=None):
    if vars_to_del:
        for v in vars_to_del:
            try:
                del v
            except Exception:
                pass
    if torch.cuda.is_available():
        try:
            if device is not None and "cuda" in str(device):
                torch.cuda.synchronize(device)
            else:
                torch.cuda.synchronize()
        except Exception:
            pass
        torch.cuda.empty_cache()
    gc.collect()


def apply_shift_to_hsi(hsi_tensor, shift_x=0, shift_y=0):
    if shift_x == 0 and shift_y == 0:
        return hsi_tensor
    
    C, H, W = hsi_tensor.shape
    shifted = torch.zeros_like(hsi_tensor)
    
    src_y_start = max(0, -shift_y)
    src_y_end = min(H, H - shift_y)
    src_x_start = max(0, -shift_x)
    src_x_end = min(W, W - shift_x)
    
    tgt_y_start = max(0, shift_y)
    tgt_y_end = min(H, H + shift_y)
    tgt_x_start = max(0, shift_x)
    tgt_x_end = min(W, W + shift_x)
    
    shifted[:, tgt_y_start:tgt_y_end, tgt_x_start:tgt_x_end] = \
        hsi_tensor[:, src_y_start:src_y_end, src_x_start:src_x_end]
    
    return shifted

def _normalize_and_augment(state_dict, target_keys):
    sd = state_dict
    if isinstance(sd, dict):
        for key in ["state_dict", "model_state_dict", "model"]:
            if key in sd and isinstance(sd[key], (dict, OrderedDict)):
                sd = sd[key]
                break
    
    cleaned = OrderedDict()
    
    def _strip_once(k):
        for p in ("module.", "_orig_mod.", "model.", "net.", "student.", "ema_model.", "unwrapped_model."):
            if k.startswith(p):
                return k[len(p):]
        return k
    
    for k, v in sd.items():
        nk = _strip_once(_strip_once(k))
        cleaned[nk] = v

    tgt = set(target_keys)
    aug = OrderedDict(cleaned)
    legacy_roots = ("patch_embed.", "layers.", "norm.", "pos_drop.", "seg_head.")
    
    if any(k.startswith("backbone.") for k in tgt):
        for k in list(cleaned.keys()):
            if k.startswith(legacy_roots):
                bk = "backbone." + k
                if bk not in aug:
                    aug[bk] = cleaned[k]
    
    if any(k.startswith("rgb_model.backbone.") for k in tgt):
        for k in list(cleaned.keys()):
            if k.startswith(legacy_roots):
                rbk = "rgb_model.backbone." + k
                if rbk not in aug:
                    aug[rbk] = cleaned[k]
        for k, v in list(cleaned.items()):
            if k.startswith("rgb_model.") and not k.startswith("rgb_model.backbone."):
                suffix = k[len("rgb_model."):]
                nk = "rgb_model.backbone." + suffix
                if nk not in aug:
                    aug[nk] = v
    
    if any(k.startswith("hsi_model.backbone.") for k in tgt):
        for k in list(cleaned.keys()):
            if k.startswith(legacy_roots):
                hbk = "hsi_model.backbone." + k
                if hbk not in aug:
                    aug[hbk] = cleaned[k]
        for k, v in list(cleaned.items()):
            if k.startswith("hsi_model.") and not k.startswith("hsi_model.backbone."):
                suffix = k[len("hsi_model."):]
                nk = "hsi_model.backbone." + suffix
                if nk not in aug:
                    aug[nk] = v

    if not any(k.startswith("backbone.") for k in tgt):
        for k, v in list(aug.items()):
            if k.startswith("backbone."):
                stripped = k[len("backbone."):]
                if stripped not in aug:
                    aug[stripped] = v
    
    return aug


def build_and_load(config_path, checkpoint_path, device_id="0"):
    device_id = str(device_id)
    
    with open(config_path, "r") as f:
        config = yaml.safe_load(f)

    for key in ['rgb_model_config_path', 'rgb_checkpoint_path',
                'hsi_model_config_path', 'hsi_checkpoint_path']:
        if key in config and not Path(config[key]).is_absolute():
            config[key] = str(project_root / config[key])

    config.setdefault("training", {})["compile_model"] = False
    config.setdefault("hardware", {})["gpu"] = device_id

    if torch.cuda.is_available():
        try:
            torch.cuda.set_device(int(device_id))
        except Exception:
            pass
        device = torch.device(f"cuda:{device_id}")
    else:
        device = torch.device("cpu")

    print(f"[GPU {device_id}] Loading dataset: {config['dataset_name']} (modality={config['modality']}) ...", flush=True)
    
    with redirect_stdout(open(os.devnull, "w")):
        train_ds, val_ds, test_ds, num_classes, class_names = create_datasets(
            config=config,
            dataset_name=config["dataset_name"],
            modality=config["modality"]
        )
    
    config["class_names"] = class_names
    print(f"[GPU {device_id}] Dataset loaded: test={len(test_ds)}, classes={num_classes}", flush=True)

    print(f"[GPU {device_id}] Loading model from checkpoint ...", flush=True)
    with redirect_stdout(open(os.devnull, "w")):
        model = build_model_finetune(config, num_classes)

    if not os.path.exists(checkpoint_path):
        raise FileNotFoundError(checkpoint_path)
    
    try:
        ckpt = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    except Exception as e:
        print(f"Error loading checkpoint: {e}")
        raise

    tgt_sd = model.state_dict()
    cand_sd = _normalize_and_augment(ckpt, tgt_sd.keys())
    to_load = OrderedDict((k, v) for k, v in cand_sd.items()
                          if k in tgt_sd and tuple(tgt_sd[k].shape) == tuple(v.shape))
    model.load_state_dict(to_load, strict=False)

    model.to(device).eval()
    print(f"[GPU {device_id}] Model ready.", flush=True)
    return model, config, test_ds, num_classes, class_names, device


def evaluate_full(
    model, config, test_ds, num_classes, device=None,
    batch_size=1, show_progress=False, device_id=None,
    clear_gpu=True, print_results=True, shift_x=0, shift_y=0
):
    if device_id is None:
        device_id = os.environ.get("EVAL_GPU", "0")
    if device_id is not None and torch.cuda.is_available():
        try:
            torch.cuda.set_device(int(device_id))
        except Exception:
            pass
        device = torch.device(f"cuda:{device_id}")
    elif device is None:
        device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

    model = model.to(device).eval()
    class_names = config.get("class_names", [f"class_{i}" for i in range(num_classes)])

    test_loader = DataLoader(
        test_ds, batch_size=batch_size, shuffle=False,
        num_workers=0, pin_memory=torch.cuda.is_available()
    )
    
    config_no_aug = deepcopy(config)
    config_no_aug.setdefault('augmentation', {})['enable'] = False
    _, _, test_dataset_for_labels, _, _ = create_datasets(
        config=config_no_aug,
        dataset_name=config['dataset_name'],
        modality=config.get('modality', 'rgb')
    )
    
    label_loader = DataLoader(
        test_dataset_for_labels, batch_size=batch_size,
        shuffle=False, num_workers=0, pin_memory=True
    )

    preds, labels = [], []
    total = len(test_loader)
    last_pct = -1
    
    if show_progress:
        print(f"[Eval] Evaluate model (shift_x={shift_x}, shift_y={shift_y})")
        print("[Eval] Progress: 0%", end="\r", flush=True)

    x = logits = pred = None
    try:
        with torch.no_grad():
            for i, (data_batch, label_batch) in enumerate(zip(test_loader, label_loader), 1):
                y = label_batch['label'].to(device, non_blocking=True)
                
                if config["modality"] == "rgb":
                    x = data_batch["image"].to(device, non_blocking=True)
                    logits = model(x)
                elif config["modality"] == "hsi":
                    x = data_batch["image"].to(device, non_blocking=True)
                    if shift_x != 0 or shift_y != 0:
                        x = torch.stack([apply_shift_to_hsi(x[b], shift_x, shift_y) for b in range(x.shape[0])])
                    logits = model(x)
                else:
                    x_rgb = data_batch["rgb"].to(device, non_blocking=True)
                    x_hsi = data_batch["hsi"].to(device, non_blocking=True)
                    if shift_x != 0 or shift_y != 0:
                        x_hsi = torch.stack([apply_shift_to_hsi(x_hsi[b], shift_x, shift_y) for b in range(x_hsi.shape[0])])
                    x = {"rgb": x_rgb, "hsi": x_hsi}
                    logits = model(x)

                if logits.shape[2:] != y.shape[1:]:
                    logits = F.interpolate(logits, size=y.shape[1:], mode="bilinear", align_corners=False)
                
                preds.append(logits.argmax(1).cpu())
                labels.append(y.cpu())

                if show_progress and total > 0:
                    pct = int(i * 100 / total)
                    if pct > last_pct:
                        print(f"[Eval] Progress: {pct}%", end="\r", flush=True)
                        last_pct = pct
        
        if show_progress:
            print("[Eval] Progress: 100%")

        all_preds = torch.cat(preds).numpy()
        all_labels = torch.cat(labels).numpy()
        m = calculate_metrics(all_preds, all_labels, num_classes)

        if print_results:
            print(f"\n[Eval] Results (shift_x={shift_x}, shift_y={shift_y})")
            print(f"[Eval] mIoU: {m['miou']:.4f}")
            print("[Eval] Per-class IoU:")
            for ci, cname in enumerate(class_names):
                print(f"  IoU[{ci} - {cname}]: {m['class_iou'][ci]:.4f}")

        return m

    finally:
        if clear_gpu:
            try:
                model.to("cpu")
            except Exception:
                pass
            del x, logits, pred
            if torch.cuda.is_available():
                try:
                    torch.cuda.synchronize()
                except Exception:
                    pass
                torch.cuda.empty_cache()
            gc.collect()


def visualize_grid(models_spec, input_idxs, device_id="0", shift_x=0, shift_y=0, 
                   title_fs=15, cell_w=3.0, cell_h=2.8):
    palette = get_palette_for_pil(7)
    rows = len(input_idxs)
    cols = len(models_spec) + 2
    
    fig, axs = plt.subplots(
        rows, cols, figsize=(cell_w * cols, cell_h * rows),
        squeeze=False, gridspec_kw={'hspace': 0.03, 'wspace': 0.05}
    )

    axs[0, 0].set_title('Input\n', fontsize=title_fs)
    axs[0, 1].set_title('GT\n', fontsize=title_fs)

    legend_class_names = None
    legend_num_classes = None

    for i, spec in enumerate(models_spec):
        print(f"Loading: {spec['name']} on GPU {device_id} (shift_x={shift_x}, shift_y={shift_y})")
        try:
            model, config, test_ds, num_classes, class_names, device = build_and_load(
                spec['config'], spec['checkpoint'], device_id=device_id
            )
            if legend_class_names is None:
                legend_class_names = class_names
                legend_num_classes = num_classes

            col = i + 2

            for r, idx in enumerate(input_idxs):
                if idx >= len(test_ds):
                    axs[r, col].text(0.5, 0.5, 'Idx OOB', ha='center', va='center',
                                     transform=axs[r, col].transAxes, fontsize=10)
                    axs[r, col].axis('off')
                    continue

                sample = test_ds[idx]
                with torch.no_grad():
                    if config["modality"] == "rgb":
                        x = sample["image"].unsqueeze(0).to(device)
                        logits = model(x)
                        vis_mod = "rgb"
                        img = sample["image"].cpu()
                    elif config["modality"] == "hsi":
                        x = sample["image"].unsqueeze(0).to(device)
                        if shift_x != 0 or shift_y != 0:
                            x = apply_shift_to_hsi(x[0], shift_x, shift_y).unsqueeze(0)
                        logits = model(x)
                        vis_mod = "hsi"
                        img = sample["image"].cpu()
                    else:
                        x_rgb = sample["rgb"].unsqueeze(0).to(device)
                        x_hsi = sample["hsi"].unsqueeze(0).to(device)
                        if shift_x != 0 or shift_y != 0:
                            x_hsi = apply_shift_to_hsi(x_hsi[0], shift_x, shift_y).unsqueeze(0)
                        x = {"rgb": x_rgb, "hsi": x_hsi}
                        logits = model(x)
                        vis_mod = "rgb"
                        img = sample["rgb"].cpu()

                    if logits.shape[2:] != sample["label"].shape:
                        logits = F.interpolate(
                            logits, size=sample["label"].shape, mode="bilinear", align_corners=False
                        )
                    pred = logits.argmax(1).squeeze().cpu()

                if i == 0:
                    img_disp = denorm_image(img, vis_mod)
                    if vis_mod == "rgb":
                        axs[r, 0].imshow(img_disp.permute(1, 2, 0).numpy())
                    else:
                        hsi_vis = (img_disp[:10].mean(0).numpy()
                                   if img_disp.ndim == 3 and img_disp.shape[0] >= 10
                                   else img_disp[0].numpy())
                        axs[r, 0].imshow(hsi_vis, cmap="gray")
                    axs[r, 0].axis("off")
                    
                    gt_p = Image.fromarray(sample["label"].numpy().astype(np.uint8), mode="P")
                    gt_p.putpalette(palette)
                    axs[r, 1].imshow(gt_p.convert("RGB"))
                    axs[r, 1].axis("off")

                pred_p = Image.fromarray(pred.numpy().astype(np.uint8), mode="P")
                pred_p.putpalette(palette)
                axs[r, col].imshow(pred_p.convert("RGB"))
                axs[r, col].axis("off")

            axs[0, col].set_title(spec['title'], fontsize=title_fs)

            try:
                model.to("cpu")
            except Exception:
                pass
            cleanup_cuda(device=device, vars_to_del=["model", "x", "logits", "pred"])
            del model

        except Exception as e:
            print(f"Error loading {spec['name']}: {e}")
            col = i + 2
            for r in range(rows):
                axs[r, col].text(0.5, 0.5, 'Error', ha='center', va='center',
                                 transform=axs[r, col].transAxes, fontsize=10)
                axs[r, col].axis('off')
            axs[0, col].set_title(f"Error: {spec['title']}", fontsize=10)

    if legend_class_names is not None:
        fig.tight_layout(pad=0.05, w_pad=0.05, h_pad=0.03)
        fig.canvas.draw()
        
        bottom_of_grid = min(ax.get_position().y0 for ax in axs.ravel())
        handles = []
        for ci in range(legend_num_classes):
            rgb = palette[ci*3:ci*3+3]
            color = tuple(v/255. for v in rgb)
            label = legend_class_names[ci] if ci < len(legend_class_names) else f"class_{ci}"
            handles.append(mpatches.Patch(color=color, label=label))
        
        gap = 0.004
        fig.legend(
            handles=handles, loc='upper center',
            bbox_to_anchor=(0.5, bottom_of_grid - gap),
            bbox_transform=fig.transFigure,
            ncol=legend_num_classes, frameon=False,
            fontsize=15, columnspacing=0.9, handlelength=1.6
        )
    else:
        fig.tight_layout(pad=0.05, w_pad=0.05, h_pad=0.03)

    plt.show()

print("All functions loaded successfully")

### Run Visualization

In [ ]:
DEVICE = os.environ.get("EVAL_GPU", "7")

VIS_CONFIGS = "/home/jon86439/BCAF_2026/ModelConfigs"
VIS_MODELS  = "/home/jon86439/BCAF_2026/models/paper"

visualization_models = [
    ("rgb_256",      f"{VIS_CONFIGS}/RGB/swin_t_rgb_256.yaml",            f"{VIS_MODELS}/swin_t_SpectralWaste_rgb_256_best.pth",            "RGB-256\n(Origin 256)"),
    ("rgb_1024",     f"{VIS_CONFIGS}/RGB/swin_t_rgb_1024.yaml",           f"{VIS_MODELS}/swin_t_SpectralWaste_rgb_1024_best.pth",          "RGB-1024\n(Origin 256)"),
    ("rgb_2048",     f"{VIS_CONFIGS}/RGB/swin_t_rgb_2048.yaml",           f"{VIS_MODELS}/swin_t_SpectralWaste_rgb_2048_best.pth",          "RGB-2048\n(Origin 256)"),
    ("hsi_5",        f"{VIS_CONFIGS}/HSI/adapted_swin_t_HSI_5.yaml",      f"{VIS_MODELS}/adapted_swin_t_SpectralWaste_hsi_5_best.pth",     "HSI-5\n"),
    ("fusion_1024",  f"{VIS_CONFIGS}/Fusion/BCAF_RGB_1024_HSI_5.yaml",    f"{VIS_MODELS}/BCAF_SpectralWaste_rgb1024_hsi5_best.pth",  "BCAF\nRGB-1024 + HSI-5"),
]

        
input_idxs = [0, 165, 79, 166, 118]

# Convert tuples to dicts for visualize_grid
vis_specs = [{'name': n, 'config': c, 'checkpoint': cp, 'title': t} for n, c, cp, t in visualization_models]
visualize_grid(vis_specs, input_idxs, device_id=DEVICE)

### Metrics

In [ ]:
DEVICE = os.environ.get("EVAL_GPU", "7")

CONFIGS = "/home/jon86439/BCAF_2026/ModelConfigs"
MODELS  = "/home/jon86439/BCAF_2026/models/paper"

models = [
    ("Swin T RGB 256",                  f"{CONFIGS}/RGB/swin_t_rgb_256.yaml",                f"{MODELS}/swin_t_SpectralWaste_rgb_256_best.pth"),
    ("Swin T RGB 512",                  f"{CONFIGS}/RGB/swin_t_rgb_512.yaml",                f"{MODELS}/swin_t_SpectralWaste_rgb_512_best.pth"),
    ("Swin T RGB 1024",                 f"{CONFIGS}/RGB/swin_t_rgb_1024.yaml",               f"{MODELS}/swin_t_SpectralWaste_rgb_1024_best.pth"),
    ("Swin T RGB 2048",                 f"{CONFIGS}/RGB/swin_t_rgb_2048.yaml",               f"{MODELS}/swin_t_SpectralWaste_rgb_2048_best.pth"),
    ("Swin T HSI 1",                    f"{CONFIGS}/HSI/swin_t_HSI_1.yaml",                  f"{MODELS}/swin_t_SpectralWaste_hsi_1_best.pth"),
    ("Adapted Swin T HSI 3",            f"{CONFIGS}/HSI/adapted_swin_t_HSI_3.yaml",          f"{MODELS}/adapted_swin_t_SpectralWaste_hsi_3_best.pth"),
    ("Adapted Swin T HSI 5",            f"{CONFIGS}/HSI/adapted_swin_T_HSI_5.yaml",          f"{MODELS}/adapted_swin_t_SpectralWaste_hsi_5_best.pth"),
    ("Adapted Swin T HSI 7",            f"{CONFIGS}/HSI/adapted_swin_t_HSI_7.yaml",          f"{MODELS}/adapted_swin_t_SpectralWaste_hsi_7_best.pth"),
    ("Adapted Swin T HSI 10",           f"{CONFIGS}/HSI/adapted_swin_t_HSI_10.yaml",         f"{MODELS}/adapted_swin_t_SpectralWaste_hsi_10_best.pth"),
    ("BCAF RGB 256 - HSI 5",            f"{CONFIGS}/Fusion/BCAF_RGB_256_HSI_5.yaml",         f"{MODELS}/BCAF_SpectralWaste_rgb256_hsi5_best.pth"),
    ("BCAF RGB 512 - HSI 5",            f"{CONFIGS}/Fusion/BCAF_RGB_512_HSI_5.yaml",         f"{MODELS}/BCAF_SpectralWaste_rgb512_hsi5_best.pth"),
    ("BCAF RGB 1024 - HSI 5",           f"{CONFIGS}/Fusion/BCAF_RGB_1024_HSI_5.yaml",        f"{MODELS}/BCAF_SpectralWaste_rgb1024_hsi5_best.pth"),
    ("Logit Fusion RGB 1024 - HSI 5",   f"{CONFIGS}/Fusion/logitfusion_RGB_1024_HSI_5.yaml", f"{MODELS}/logitfusion_SpectralWaste_rgb1024_hsi5_best.pth"),
]

all_results = {}

print("=" * 80)
print("EVALUATION")
print("=" * 80)

for name, config_path_str, checkpoint_path_str in models:
    print(f"\n{'=' * 80}")
    print(f"MODEL: {name}")
    print(f"{'=' * 80}")

    config_path = Path(config_path_str)
    checkpoint_path = Path(checkpoint_path_str)

    if not config_path.exists():
        print(f"  Skipping: config not found: {config_path}")
        all_results[name] = None
        continue
    if not checkpoint_path.exists():
        print(f"  Skipping: checkpoint not found: {checkpoint_path}")
        all_results[name] = None
        continue

    model, config, test_ds, num_classes, class_names, device = build_and_load(
        str(config_path), str(checkpoint_path), device_id=DEVICE
    )

    res = evaluate_full(
        model, config, test_ds, num_classes, device,
        batch_size=1, show_progress=True, print_results=True
    )

    all_results[name] = {"res": res, "class_names": class_names} if res and 'miou' in res else None

    try:
        model.to("cpu")
    except:
        pass
    cleanup_cuda(device=device, vars_to_del=["model"])
    del model

print(f"\n\n{'=' * 100}")
print("FINAL SUMMARY")
print(f"{'=' * 100}")

for name, entry in all_results.items():
    if not entry:
        print(f"\n{name}: Failed")
        continue
    res = entry["res"]
    cnames = entry["class_names"]
    print(f"\n{name}")
    print(f"  mIoU: {res['miou']:.4f}")
    if 'accuracy' in res:
        print(f"  Accuracy: {res['accuracy']:.4f}")
    if 'f1' in res:
        print(f"  F1 (Macro): {res['f1']:.4f}")
    elif 'f1_macro' in res:
        print(f"  F1 (Macro): {res['f1_macro']:.4f}")
    if 'class_iou' in res:
        for ci, iou in enumerate(res['class_iou']):
            cname = cnames[ci] if ci < len(cnames) else f"class_{ci}"
            print(f"    {cname}: {iou:.4f}")

print(f"\n{'=' * 100}\n")